# Hybrid Lorenz 63 path: burn-in + true solver vs DM

1. **Burn-in** — integrate the true Lorenz 63 solver until spin-up.
2. **Fork** — same post-burn-in state for two rollouts:
   - continue with the **true L63 solver**;
   - continue with **conditional diffusion** (autoregressive next-step prediction).
3. **Compare** — RMSE, x/y/z time series, phase portraits, animation.

Classic DA-benchmark physics: `sigma=10`, `rho=28`, `beta=8/3`, `dt=0.01`.

Set `NETWORK_PKL` and `HISTORY` to match training (`lorenz63_path_next`, 1D U-Net).

In [31]:
# --- User configuration ---
import os

from project_utils.paths import PROJECT_ROOT, CKPT_DIR, dataset_trajectory_dir

L63_CONFIG = os.path.join(PROJECT_ROOT, "configs/turbulence/lorenz63_default.yaml")
DATA_ROOT = "LORENZ63"
TRAJECTORY_SUBDIR = "full_path"

# Path to trained checkpoint (network-snapshot-*.pkl from train_cond.py)
NETWORK_PKL = os.path.join(
    CKPT_DIR,
    "l63.pkl",
)

DEVICE = "cuda:0" if __import__("torch").cuda.is_available() else "cpu"
SEED = 0

STATE_DIM = 3  # must match checkpoint img_res_h
T_BURN_IN = 2000
T_ROLLOUT = 500
OUTPUT_INTERVAL = 0.01  # match generator YAML / trained DM stride

HISTORY = 1
VAR_SELECT = [0]

NUM_STEPS = 18
SEED_DM = 0

VAR_LABELS = ["x", "y", "z"]
X0 = [0.0, 1.0, 0.0]  # classic Lorenz IC

In [32]:
import pickle
import time

import dnnlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from project_utils.normalization import (
    denormalize_tensor,
    load_norm_stats,
    normalize_tensor,
    resolve_norm_stats_path,
)
from project_utils.yaml_simple import load_simple_yaml
from turbulence.lorenz63 import Lorenz63Params, Lorenz63Solver

device = torch.device(DEVICE)
trajectory_dir = dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR)
norm_stats_path = resolve_norm_stats_path(trajectory_dir, None)
print("Trajectory dir:", trajectory_dir)
print("norm_stats:", norm_stats_path)
print("device:", device)

Trajectory dir: /home/neil/glocal-condiff-turb/datasets/LORENZ63/full_path
norm_stats: /home/neil/glocal-condiff-turb/datasets/LORENZ63/full_path/norm_stats.pt
device: cuda:0


In [33]:
def edm_sampler(
    net,
    latents,
    conditions,
    class_labels=None,
    randn_like=torch.randn_like,
    num_steps=18,
    sigma_min=0.002,
    sigma_max=80,
    rho=7,
    S_churn=0,
    S_min=0,
    S_max=float("inf"),
    S_noise=1,
):
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
    t_steps = (
        sigma_max ** (1 / rho)
        + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])])

    x_next = latents.to(torch.float64) * t_steps[0]
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])):
        x_cur = x_next
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat**2 - t_cur**2).sqrt() * S_noise * randn_like(x_cur)

        denoised = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
        d_cur = (x_hat - denoised) / t_hat
        x_next = x_hat + (t_next - t_hat) * d_cur

        if i < num_steps - 1:
            denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
            d_prime = (x_next - denoised) / t_next
            x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)
    return x_next


class StackedRandomGenerator:
    def __init__(self, device, seeds):
        self.generators = [
            torch.Generator(device).manual_seed(int(s) % (1 << 32)) for s in seeds
        ]

    def randn(self, size, **kwargs):
        return torch.stack(
            [torch.randn(size[1:], generator=g, **kwargs) for g in self.generators]
        )

    def randn_like(self, input):
        return self.randn(input.shape, dtype=input.dtype, layout=input.layout, device=input.device)


def frame_tensor(x: torch.Tensor) -> torch.Tensor:
    # Map state (3,) to dataloader layout (1, 3, 1).
    return x.reshape(1, STATE_DIM, 1)


def make_initial_condition(x0, *, device, dtype: torch.dtype) -> torch.Tensor:
    return torch.tensor(list(x0), device=device, dtype=dtype)


def build_condition(frames_phys, history: int, stats: dict) -> torch.Tensor:
    recent = frames_phys[-history:]
    normed = [normalize_tensor(f, stats) for f in recent]
    block = torch.stack(normed, dim=0)
    C, H, W = block.shape[1], block.shape[2], block.shape[3]
    cond = block.reshape(history * C, H, W)
    return cond.unsqueeze(0)


def to_seq(traj: torch.Tensor) -> torch.Tensor:
    # (T, 1, 3, 1) -> (T, 3) for plotting.
    return traj[:, 0, :, 0]

## Phase 1: Lorenz 63 solver burn-in

Integrate the true model for `T_BURN_IN` saved frames (same spacing as `generate_fields.py`), then keep the state for the DM fork.

In [34]:
cfg = load_simple_yaml(L63_CONFIG)
l63_cfg = dict(cfg.get("lorenz63", {}))
l63_cfg["device"] = str(device)

params = Lorenz63Params(**l63_cfg)
params.validate()
timesteps_per_output = max(1, int(round(OUTPUT_INTERVAL / params.dt)))

dtype = torch.float32 if params.precision == "single" else torch.float64
x0 = make_initial_condition(X0, device=device, dtype=dtype)
solver = Lorenz63Solver(x0, **params.to_core_kwargs())

path_burn = [frame_tensor(solver.fields()["x"].detach().cpu())]
for _ in tqdm(range(T_BURN_IN), desc="L63 burn-in"):
    solver.step(timesteps_per_output)
    frame = frame_tensor(solver.fields()["x"].detach().cpu())
    if not torch.isfinite(frame).all():
        raise RuntimeError("Non-finite values during burn-in")
    path_burn.append(frame)

state_after_burn = {k: v.clone() for k, v in solver.get_state().items()}
path_burn = torch.stack(path_burn, dim=0)

print(f"Burn-in: {path_burn.shape[0]} frames")
print(f"Last frame (x,y,z): {path_burn[-1, 0, :, 0].tolist()}")

L63 burn-in: 100%|██████████| 2000/2000 [00:01<00:00, 1187.65it/s]

Burn-in: 2001 frames
Last frame (x,y,z): [-1.272167682647705, -2.244138240814209, 12.332416534423828]


## Phase 2: Load trained conditional diffusion model

In [ ]:
if not os.path.isfile(NETWORK_PKL):
    raise FileNotFoundError(
        f"Set NETWORK_PKL to your training snapshot. Not found: {NETWORK_PKL}\n"
        "Train with: torchrun scripts/train/train_cond.py runs=lorenz63_path_next"
    )

norm_stats = load_norm_stats(norm_stats_path)
print(f"Loaded norm_stats: vmin={norm_stats['vmin']:.6g}, vmax={norm_stats['vmax']:.6g}")

print(f"Loading network from {NETWORK_PKL} ...")
with dnnlib.util.open_url(NETWORK_PKL, verbose=True) as f:
    ckpt = pickle.load(f)
net = ckpt["ema"].to(device).eval()

n_target = len(VAR_SELECT)
n_cond = HISTORY * len(VAR_SELECT)
print(f"Model: {net.model_name}, target_ch={net.img_input_channels}, cond_ch={net.img_cond_channels}")
print(f"Resolution: H={net.img_res_h}, W={net.img_res_w}")
assert net.img_input_channels == n_target, "VAR_SELECT / checkpoint mismatch"
assert net.img_cond_channels == n_cond, "HISTORY / checkpoint mismatch"
assert net.img_res_h == STATE_DIM, f"STATE_DIM={STATE_DIM} must match img_res_h={net.img_res_h}"
assert net.img_res_w == 1, "Expected img_res_w=1 for 1D L63 layout"

## Phase 3: Parallel rollout after burn-in

From the **same** post-burn-in state:
- **True solver** — continue integrating L63 for `T_ROLLOUT` saved frames.
- **DM** — autoregressive conditional diffusion for `T_ROLLOUT` frames.

In [ ]:
if path_burn.shape[0] < HISTORY:
    raise ValueError(f"Need at least HISTORY={HISTORY} burn-in frames")

core_kw = params.to_core_kwargs()

x_fork = path_burn[-1][0, :, 0].to(device=device, dtype=dtype)
solver_cont = Lorenz63Solver(x_fork, **core_kw)
solver_cont.set_state(state_after_burn)

rollout_solver = []
for _ in tqdm(range(T_ROLLOUT), desc="True solver rollout"):
    solver_cont.step(timesteps_per_output)
    frame = frame_tensor(solver_cont.fields()["x"].detach().cpu())
    if not torch.isfinite(frame).all():
        raise RuntimeError("Non-finite values during solver rollout")
    rollout_solver.append(frame)
path_solver = torch.stack(rollout_solver, dim=0)

path_dm_hist = [path_burn[i].clone() for i in range(path_burn.shape[0])]
rnd = StackedRandomGenerator(device, [SEED_DM])

rollout_dm = []
t0 = time.time()
for step in tqdm(range(T_ROLLOUT), desc="DM rollout"):
    cond = build_condition(path_dm_hist, HISTORY, norm_stats).to(device)
    latents = rnd.randn([1, n_target, STATE_DIM, 1], device=device)
    with torch.no_grad():
        sample_norm = edm_sampler(net, latents, cond, num_steps=NUM_STEPS).to(torch.float32)
    next_phys = denormalize_tensor(sample_norm[0].cpu(), norm_stats)
    if not torch.isfinite(next_phys).all():
        raise RuntimeError(f"Non-finite DM sample at step {step}")
    rollout_dm.append(next_phys)
    path_dm_hist.append(next_phys)

path_dm = torch.stack(rollout_dm, dim=0)
print(f"DM rollout done in {time.time() - t0:.1f}s")
print("Shapes:", path_solver.shape, path_dm.shape)

## Phase 4: Compare true solver vs DM

In [ ]:
solver_seq = to_seq(path_solver)
dm_seq = to_seq(path_dm)

diff = solver_seq - dm_seq
rmse = diff.pow(2).mean(dim=1).sqrt()
rel_rmse = rmse / solver_seq.pow(2).mean(dim=1).sqrt().clamp_min(1e-8)

steps = np.arange(T_ROLLOUT)
fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axs[0].plot(steps, rmse.numpy(), color="C0")
axs[0].set_ylabel("RMSE (physical)")
axs[0].set_title("True solver vs DM — error after shared burn-in")
axs[0].grid(True, alpha=0.3)
axs[1].plot(steps, rel_rmse.numpy(), color="C1")
axs[1].set_ylabel("relative RMSE")
axs[1].set_xlabel("rollout step k (after burn-in)")
axs[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean RMSE over rollout: {rmse.mean().item():.4g}")
print(f"Final-step RMSE: {rmse[-1].item():.4g}")

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for d, lab in enumerate(VAR_LABELS):
    axs[d].plot(solver_seq[:, d].numpy(), label="true solver", linewidth=1.2)
    axs[d].plot(dm_seq[:, d].numpy(), label="DM", linewidth=1.2, alpha=0.85)
    axs[d].set_ylabel(lab)
    axs[d].grid(True, alpha=0.3)
    axs[d].legend(loc="upper right", fontsize=8)
axs[0].set_title("x, y, z — true solver vs DM")
axs[-1].set_xlabel("rollout step k")
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 4))
ax0 = fig.add_subplot(1, 2, 1, projection="3d")
ax0.plot(
    solver_seq[:, 0].numpy(),
    solver_seq[:, 1].numpy(),
    solver_seq[:, 2].numpy(),
    linewidth=0.5,
    label="true",
)
ax0.plot(
    dm_seq[:, 0].numpy(),
    dm_seq[:, 1].numpy(),
    dm_seq[:, 2].numpy(),
    linewidth=0.5,
    alpha=0.8,
    label="DM",
)
ax0.set_xlabel("x")
ax0.set_ylabel("y")
ax0.set_zlabel("z")
ax0.set_title("3D attractor")
ax0.legend()

ax1 = fig.add_subplot(1, 2, 2)
ax1.plot(solver_seq[:, 0].numpy(), solver_seq[:, 2].numpy(), linewidth=0.5, label="true")
ax1.plot(dm_seq[:, 0].numpy(), dm_seq[:, 2].numpy(), linewidth=0.5, alpha=0.8, label="DM")
ax1.set_xlabel("x")
ax1.set_ylabel("z")
ax1.set_title("x–z projection")
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()